In [1]:
import argparse
import sys
import os
import numpy as np
import face_recognition
import chromadb
from chromadb.config import Settings
from PIL import Image

/home/ai_vison/miniconda3/envs/ppe/lib/python3.12/site-packages/face_recognition_models/__init__.py:7: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename


In [3]:
DB_PATH = "./face_vector_db"         
COLLECTION_NAME = "employee_faces"
 
def get_collection() -> chromadb.Collection:
    """Return (or create) the ChromaDB collection."""
    client = chromadb.PersistentClient(path=DB_PATH)
    print(os.path.abspath('./face_vector_db'))
    collection = client.get_or_create_collection(
        name=COLLECTION_NAME,
        metadata={"hnsw:space": "cosine"},   # cosine similarity
    )
    return collection

In [4]:
def load_image_rgb(image_path: str) -> np.ndarray:
    """Load an image as an RGB numpy array (handles RGBA, palette, etc.)."""
    img = Image.open(image_path).convert("RGB")
    return np.array(img)

In [5]:
def extract_embedding(image_path: str) -> list[float] | None:
    """
    Detect the first face in the image and return its 128-d embedding.
    Returns None if no face is found.
    """
    img = load_image_rgb(image_path)
 
    # Use CNN model for better accuracy; use "hog" for CPU-only speed
    face_locations = face_recognition.face_locations(img, model="hog")
 
    if not face_locations:
        print(f"  [!] No face detected in '{image_path}'.")
        return None
 
    if len(face_locations) > 1:
        print(f"  [!] Multiple faces detected in '{image_path}'; using the first one.")
 
    encodings = face_recognition.face_encodings(img, known_face_locations=face_locations)
    return encodings[0].tolist()          # 128-dimensional list[float]


In [6]:
def enroll_employee(image_path: str, employee_id: str) -> None:
    """Extract face embedding and store it in the vector DB."""
    print(f"[Enroll] employee_id={employee_id}, image={image_path}")
 
    embedding = extract_embedding(image_path)
    if embedding is None:
        print("  [!] Enrollment aborted — no face found.")
        return
 
    collection = get_collection()
 
    # ChromaDB requires a unique string ID per document.
    # Using employee_id as the ID means re-enrolling the same ID updates it.
    collection.upsert(
        ids=[employee_id],
        embeddings=[embedding],
        metadatas=[{"employee_id": employee_id, "source_image": image_path}],
    )
    print(f"  [✓] Enrolled employee '{employee_id}' successfully.")

In [10]:
enroll_employee("/home/ai_vison/Desktop/PPE/model_pipeline/siyathma.jpeg", "emp004")

[Enroll] employee_id=emp004, image=/home/ai_vison/Desktop/PPE/model_pipeline/siyathma.jpeg
/home/ai_vison/Desktop/PPE/model_pipeline/face_vector_db
  [✓] Enrolled employee 'emp004' successfully.


In [11]:
def identify_person(image_path: str, top_k: int = 1, threshold: float = 0.4) -> None:
    """
    Extract face from image, query the vector DB, and print the best match.
 
    threshold: cosine *distance* threshold (0 = identical, 1 = opposite).
               Values below this are considered a match.
               Typical good value: 0.35–0.45.
    """
    print(f"[Identify] image={image_path}")
 
    embedding = extract_embedding(image_path)
    if embedding is None:
        print("  [!] Identification aborted — no face found.")
        return
 
    collection = get_collection()
 
    if collection.count() == 0:
        print("  [!] The database is empty. Enroll some employees first.")
        return
 
    results = collection.query(
        query_embeddings=[embedding],
        n_results=min(top_k, collection.count()),
        include=["metadatas", "distances"],
    )
 
    matches     = results["metadatas"][0]
    distances   = results["distances"][0]
 
    print(f"\n  Top-{top_k} result(s):")
    for rank, (meta, dist) in enumerate(zip(matches, distances), start=1):
        similarity = 1 - dist          # cosine similarity (higher = more similar)
        status = "✓ MATCH" if dist < threshold else "✗ NO MATCH"
        print(f"  [{rank}] employee_id={meta['employee_id']}  "
              f"similarity={similarity:.4f}  distance={dist:.4f}  {status}")
 
    best_dist = distances[0]
    if best_dist < threshold:
        print(f"\n  => Identified as: {matches[0]['employee_id']}")
    else:
        print("\n  => Person NOT found in the database (below similarity threshold).")


In [15]:
identify_person("/home/ai_vison/Desktop/PPE/model_pipeline/kanishka_test1.jpg")

[Identify] image=/home/ai_vison/Desktop/PPE/model_pipeline/kanishka_test1.jpg
/home/ai_vison/Desktop/PPE/model_pipeline/face_vector_db

  Top-1 result(s):
  [1] employee_id=emp002  similarity=0.9562  distance=0.0438  ✓ MATCH

  => Identified as: emp002
